# AROsearch — Indexing Pipeline

Builds the FAISS dense index, BM25 sparse index, and metadata parquet for AROsearch.
Runs end-to-end on a free Colab T4 GPU in ~5 minutes.

**What you do:**
1. Runtime → Change runtime type → T4 GPU
2. Run all cells top to bottom
3. The final cell zips the artifacts and downloads them to your machine
4. Add the zip contents to your `arosearch/data/` folder and commit to GitHub

**Data:** CARD's ARO ontology, CC-BY 4.0, fetched fresh from card.mcmaster.ca.

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu bm25s pyarrow

## 2. Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## 3. Download CARD data

ARO is CC-BY 4.0. We pull the latest release from CARD's distribution endpoint.

In [ ]:
!mkdir -p data
!wget -q https://card.mcmaster.ca/latest/data -O data/card.tar.bz2
!tar -xjf data/card.tar.bz2 -C data/
!ls -la data/

## 4. Parse CARD JSON

One row per unique ARO term with the fields we need for the two text views.

In [ ]:
import json
import pandas as pd

with open("data/card.json") as f:
    raw = json.load(f)

records = []
for model_key, model in raw.items():
    if not isinstance(model, dict) or not model_key.isdigit():
        continue
    if "ARO_accession" not in model:
        continue
    
    drug_classes, mechanisms, families = [], [], []
    for cat in (model.get("ARO_category") or {}).values():
        cls = cat.get("category_aro_class_name", "")
        cname = cat.get("category_aro_name", "")
        if cls == "Drug Class":
            drug_classes.append(cname)
        elif cls == "Resistance Mechanism":
            mechanisms.append(cname)
        elif cls == "AMR Gene Family":
            families.append(cname)
    
    records.append({
        "aro_id": f"ARO:{model['ARO_accession']}",
        "name": model.get("ARO_name", ""),
        "description": model.get("ARO_description", ""),
        "drug_classes": "; ".join(drug_classes),
        "mechanisms": "; ".join(mechanisms),
        "families": "; ".join(families),
    })

df = pd.DataFrame(records).drop_duplicates(subset="aro_id").reset_index(drop=True)
print(f"Loaded {len(df)} unique ARO entries")
df.head(3)

## 5. Build the two text views per ARO entry

- **Lexical view** (for BM25): name + gene family. Term-dense, good for exact matching.
- **Semantic view** (for embeddings): name + description + drug class + mechanism. Prose-shaped.

In [ ]:
lexical_view = (df["name"] + " " + df["families"]).tolist()

semantic_view = (
    df["name"] + ". "
    + df["description"] + " "
    + "Drug classes: " + df["drug_classes"] + ". "
    + "Mechanisms: " + df["mechanisms"] + "."
).tolist()

print("Lexical sample:", lexical_view[0][:200])
print()
print("Semantic sample:", semantic_view[0][:300])

## 6. Build dense (FAISS) index on GPU

Qwen3-Embedding-0.6B, fp16 on T4. ~30 seconds for ~6K docs.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
EMBED_DIM = 1024

print(f"Loading {EMBED_MODEL} on GPU...")
model = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device="cuda")
model = model.half()  # fp16 — faster on T4, ~half VRAM

print(f"Encoding {len(semantic_view)} documents...")
vecs = model.encode(
    semantic_view,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

assert vecs.shape == (len(df), EMBED_DIM), f"unexpected shape {vecs.shape}"

faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(vecs)
print(f"FAISS index: {faiss_index.ntotal} vectors, dim {EMBED_DIM}")

faiss.write_index(faiss_index, "data/aro_index.faiss")
print("Saved data/aro_index.faiss")

## 7. Build sparse (BM25S) index

In [ ]:
import pickle
import bm25s
from bm25s import BM25

tokens = bm25s.tokenize(lexical_view, stopwords="en")
bm25 = BM25()
bm25.index(tokens)

with open("data/aro_bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)
print("Saved data/aro_bm25.pkl")

## 8. Save metadata

In [ ]:
df.to_parquet("data/aro_meta.parquet", index=False)
print(f"Saved data/aro_meta.parquet  ({len(df)} rows)")

## 9. Smoke test the indices before downloading

Run a few queries to confirm everything works before we commit anything to git.

In [ ]:
SIGMOID_A, SIGMOID_B = 10.0, 0.5

def search(query, top_k=5):
    # Dense
    qvec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    cosim_scores, cosim_idx = faiss_index.search(qvec, 50)
    dense = {int(i): float(s) for i, s in zip(cosim_idx[0], cosim_scores[0]) if i >= 0}
    
    # Sparse
    qtokens = bm25s.tokenize([query], stopwords="en")
    results, scores = bm25.retrieve(qtokens, k=50)
    sparse_raw = {int(i): float(s) for i, s in zip(results[0], scores[0])}
    
    # Normalize BM25 to [0,1]
    if sparse_raw:
        vals = np.array(list(sparse_raw.values()))
        lo, hi = vals.min(), vals.max()
        sparse = {k: float((v - lo) / (hi - lo + 1e-9)) for k, v in sparse_raw.items()} if hi > lo else {k: 1.0 for k in sparse_raw}
    else:
        sparse = {}
    
    # Fuse
    candidates = set(dense) | set(sparse)
    hits = []
    for idx in candidates:
        cos = max(0.0, dense.get(idx, 0.0))
        bm = sparse.get(idx, 0.0)
        alpha = 1.0 / (1.0 + np.exp(-SIGMOID_A * (bm - SIGMOID_B)))
        score = alpha * bm + (1 - alpha) * cos
        hits.append((idx, score, alpha, bm, cos))
    
    hits.sort(key=lambda x: (x[1], x[3]), reverse=True)
    return hits[:top_k]

test_queries = [
    "enzymes that hydrolyze last-resort beta-lactams",
    "KPC-2",
    "efflux pump fluoroquinolone resistance",
    "vancomycin resistance enterococci",
]

for q in test_queries:
    print(f"\n=== {q} ===")
    for idx, score, alpha, bm, cos in search(q):
        row = df.iloc[idx]
        print(f"  {row['aro_id']:14}  score={score:.3f}  α={alpha:.2f}  cos={cos:.2f}  bm25={bm:.2f}  {row['name'][:60]}")

## 10. Inspect artifact sizes

Just so we know what we're committing. FAISS index for ~6K × 1024 fp32 vectors is ~26 MB — fine for git.

In [ ]:
!ls -lh data/aro_index.faiss data/aro_bm25.pkl data/aro_meta.parquet

## 11. Bundle and download

Zips the three artifacts, then triggers a browser download. Drop the contents into your local `arosearch/data/` and commit.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("arosearch_artifacts", "zip", "data",
                    base_dir=None)
# Filter to just the three artifacts (not card.json or the tar.bz2)
import zipfile
with zipfile.ZipFile("arosearch_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("data/aro_index.faiss", "aro_index.faiss")
    z.write("data/aro_bm25.pkl", "aro_bm25.pkl")
    z.write("data/aro_meta.parquet", "aro_meta.parquet")

print("Bundle ready. Starting download...")
files.download("arosearch_artifacts.zip")